# Loan Approval Prediction - Model Training

This notebook contains the pipeline to load, preprocess, train, and evaluate a Random Forest model on the Loan Approval dataset, saving the trained pipeline for later use.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [ ]:
# ==========================================
# 1. LOAD DATASET
# ==========================================
# Using the relative path to load the dataset
csv_path = "../loan_approval_dataset.csv"
df = pd.read_csv(csv_path)
df.head()

In [ ]:
# ==========================================
# 2. CLEAN COLUMN NAMES & CATEGORICAL VALUES
# ==========================================
# Strip leading/trailing spaces from column names
df.columns = df.columns.str.strip()

# Strip leading/trailing spaces from string values in categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    df[col] = df[col].str.strip()

df.info()

In [ ]:
# ==========================================
# 3. DEFINE FEATURES AND TARGET
# ==========================================
# Drop loan_id as it is not a feature
X = df.drop(columns=["loan_id", "loan_status"])
y = df["loan_status"].map({"Approved": 1, "Rejected": 0})

# Identify numeric and categorical columns
numeric_features = [
    "no_of_dependents", "income_annum", "loan_amount", "loan_term", "cibil_score", 
    "residential_assets_value", "commercial_assets_value", "luxury_assets_value", "bank_asset_value"
]
categorical_features = ["education", "self_employed"]

In [ ]:
# ==========================================
# 4. BUILD PREPROCESSING AND PIPELINE
# ==========================================
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ]
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced"))
])

In [ ]:
# ==========================================
# 5. TRAIN AND EVALUATE MODEL
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# ==========================================
# 6. SAVE TRAINED PIPELINE
# ==========================================
model_dir = "../model"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "loan_model.pkl")
joblib.dump(pipeline, model_path)
print(f"Saved trained model pipeline to: {model_path}")